### DistilBERT Baseline Model

In [1]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, pipeline 
import pandas as pd
from scipy import stats
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, accuracy_score

/Users/sherylcheng/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
reviews_df = pd.read_csv("final_yelp_dataset.csv")
reviews_df = reviews_df[reviews_df['tier'] != '$$$$']
reviews_df.head(5)

,business_id,name,cuisine,city,state,total_business_stars,review_count,categories,tier,user_id,individual_stars,text,date
0,YB26JvvGS2LgkxEKOObSAw,Unagi & Sushi,Japanese,Metairie,LA,4.0,62,"['Sushi Bars', 'Restaurants', 'Japanese']",$$,iAD32p6h32eKDVxsPHSRHA,5.0,i've been eating at this restaurant for over 5...,2021-01-08 01:49:36
1,jfIwOEXcVRyhZjM4ISOh4g,Oreland Pizza,American,Oreland,PA,3.0,29,"['Sandwiches', 'Restaurants', 'Burgers', 'Pizza']",$$,rYvWv-Ny16b1lMcw1IP7JQ,1.0,how does a delivery person from here get lost ...,2021-01-02 00:19:00
2,yE1raqkLX7OZsjmX3qKIKg,Butcher & Bee,American,Nashville,TN,4.0,863,"['Middle Eastern', 'American (New)', 'Restaura...",$$,j4qNLF-VNRF2DwBkUENW-w,5.0,two words: whipped. feta. \nexplosion of amazi...,2021-01-27 23:28:03
3,3PlpoDgeAQAreL8FM2LelA,Kauboi Izakaya,Japanese,Reno,NV,4.0,312,"['Sushi Bars', 'Pubs', 'Gastropubs', 'Japanese...",$$,h41RIr5Rtkq7EoJ0tU0wgQ,4.0,place was great as well as parking. \nfood was...,2021-01-30 03:15:20
4,Rv6P37KiiuowrXti2JHZNQ,Nuevo Vallarta Authentic Mexican Food,Mexican,Pinellas Park,FL,4.0,106,"['Event Planning & Services', 'Chicken Wings',...",$,8_yoGifxsLHLRPtyDgMxhw,5.0,got this to go and it is for sure authentic! t...,2021-03-11 17:17:46


In [3]:
# DistilBERT sentiment pipeline
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0  
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6790.81it/s]


In [4]:
reviews_df["cuisine"].value_counts()

cuisine
American    141523
Mexican      28866
Italian      27174
Japanese     19026
Chinese      14310
Thai          5369
Indian        4901
Greek         3639
Korean        2386
French        2319
Name: count, dtype: int64

In [ ]:
"""
SAMPLE_PER_GROUP = 5000

parts = []
for _, g in reviews_df.groupby(['cuisine', 'tier']):
    n = min(len(g), SAMPLE_PER_GROUP)
    parts.append(g.sample(n=n, random_state=42))

sampled_df = pd.concat(parts, ignore_index=True)
print(sampled_df.groupby(['cuisine', 'tier']).size().unstack(fill_value=0))
"""

tier         $    $$   $$$
cuisine                   
American  5000  5000  5000
Chinese   3706  5000   177
French     157  1307   855
Greek      959  2582    98
Indian     300  4541    60
Italian   2950  5000  2769
Japanese   456  5000  1302
Korean     225  2098    63
Mexican   5000  5000   177
Thai       258  5000    88


In [ ]:
# sample per cuisine and tier 
SAMPLE_PER_GROUP = 5000

sampled_df = (
    reviews_df.groupby(['cuisine', 'tier'], group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_GROUP), random_state=42))
    .reset_index(drop=True)
)

print(sampled_df.groupby(['cuisine', 'tier']).size().unstack(fill_value=0))
print(f"\nTotal reviews: {len(sampled_df)}")

In [6]:
def get_sentiment(texts, batch_size=128):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size].tolist()
        outputs = sentiment_model(batch, truncation=True, max_length=512)
        results.extend(outputs)
    return results

results = get_sentiment(sampled_df['text'], batch_size=128)

sampled_df['distilbert_label'] = [r['label'] for r in results]
sampled_df['distilbert_score'] = [r['score'] for r in results]

In [7]:
# apply threshold from CV results
best_thresh = 0.30 

sampled_df['raw_positive_score'] = [
    r['score'] if r['label'] == 'POSITIVE' else 1 - r['score']
    for r in results
]
sampled_df['distilbert_sentiment'] = (
    sampled_df['raw_positive_score'] >= best_thresh
).astype(float)

print(f"Applied threshold: {best_thresh}")

Applied threshold: 0.3


In [8]:
# Aggregate by cuisine & tier
MIN_REVIEWS = 50

def agg_sentiment(group):
    scores = group['distilbert_sentiment']
    n = len(group)
    if n < MIN_REVIEWS:
        return None
    sem = stats.sem(scores)
    ci = stats.t.interval(0.95, df=n-1, loc=scores.mean(), scale=sem)
    return pd.Series({
        "n_reviews":      n,
        "mean_sentiment": round(scores.mean(), 4),
        "ci_lower":       round(ci[0], 4),
        "ci_upper":       round(ci[1], 4),
        "pct_positive":   round((group['distilbert_label'] == 'POSITIVE').mean() * 100, 2),
    })

by_cuisine = (
    sampled_df.groupby('cuisine')
    .apply(agg_sentiment)
    .dropna()
    .sort_values('mean_sentiment', ascending=False)
)

by_cuisine_tier = (
    sampled_df.groupby(['cuisine', 'tier'])
    .apply(agg_sentiment)
    .dropna()
    .sort_values(['tier', 'mean_sentiment'], ascending=[True, False])
)
display(by_cuisine)
display(by_cuisine_tier)


,n_reviews,mean_sentiment,ci_lower,ci_upper,pct_positive
cuisine,,,,,
Thai,5346.0,0.7899,0.7790,0.8009,78.34
French,2319.0,0.7822,0.7654,0.7990,77.58
Korean,2386.0,0.7804,0.7638,0.7970,77.49
Indian,4901.0,0.7760,0.7643,0.7876,77.11
Greek,3639.0,0.7527,0.7387,0.7667,74.55
Japanese,6758.0,0.7351,0.7246,0.7457,72.85
Italian,10719.0,0.7215,0.7130,0.7300,71.47
Mexican,10177.0,0.7109,0.7021,0.7197,70.46
Chinese,8883.0,0.6814,0.6717,0.6911,67.33


,,n_reviews,mean_sentiment,ci_lower,ci_upper,pct_positive
cuisine,tier,,,,,
French,$,157.0,0.8726,0.8199,0.9253,86.62
Korean,$,225.0,0.8356,0.7867,0.8844,82.67
Thai,$,258.0,0.8023,0.7534,0.8512,80.23
Japanese,$,456.0,0.7654,0.7263,0.8044,75.88
Greek,$,959.0,0.7539,0.7266,0.7812,74.45
Indian,$,300.0,0.7400,0.6901,0.7899,74.00
Mexican,$,5000.0,0.7052,0.6926,0.7178,69.90
Italian,$,2950.0,0.6637,0.6467,0.6808,65.46
Chinese,$,3706.0,0.6382,0.6227,0.6536,62.95


In [9]:
cuisine_groups = [
    g['distilbert_sentiment'].values
    for _, g in sampled_df.groupby('cuisine')
    if len(g) >= MIN_REVIEWS
]
f_stat, p_val = stats.f_oneway(*cuisine_groups)
print(f"ANOVA: F = {f_stat:.2f}, p = {p_val:.4f}")

american = sampled_df[sampled_df['cuisine'] == 'American']['distilbert_sentiment'].values

for cuisine, group in sampled_df.groupby('cuisine'):
    if cuisine == 'American' or len(group) < MIN_REVIEWS:
        continue
    other = group['distilbert_sentiment'].values
    t, p = stats.ttest_ind(american, other)
    diff = other.mean() - american.mean()
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    print(f"American vs {cuisine:12s}:  diff={diff:+.4f},  t={t:.2f},  p={p:.4f}  {sig}")

r, p = stats.pearsonr(sampled_df['distilbert_sentiment'], sampled_df['individual_stars'])
print(f"\nPearson r = {r:.4f}, p = {p:.4e}")

ANOVA: F = 81.32, p = 0.0000
American vs Chinese     :  diff=+0.0296,  t=-4.68,  p=0.0000  ***
American vs French      :  diff=+0.1304,  t=-12.48,  p=0.0000  ***
American vs Greek       :  diff=+0.1009,  t=-11.67,  p=0.0000  ***
American vs Indian      :  diff=+0.1242,  t=-16.32,  p=0.0000  ***
American vs Italian     :  diff=+0.0697,  t=-11.86,  p=0.0000  ***
American vs Japanese    :  diff=+0.0833,  t=-12.21,  p=0.0000  ***
American vs Korean      :  diff=+0.1286,  t=-12.46,  p=0.0000  ***
American vs Mexican     :  diff=+0.0591,  t=-9.85,  p=0.0000  ***
American vs Thai        :  diff=+0.1381,  t=-18.88,  p=0.0000  ***

Pearson r = 0.8243, p = 0.0000e+00


### Export DistilBERT Results

In [10]:
# pick DistilBERT score column
score_candidates = ["distilbert_sentiment", "raw_positive_score", "distilbert_score", "sentiment_score", "bert_score"]
score_col = next(c for c in score_candidates if c in sampled_df.columns)

# standardize names for export
export_df = sampled_df.rename(columns={"cuisine": "cuisine_type", "individual_stars": "stars"}).copy()

# overall results by cuisine
overall = (
    export_df.groupby("cuisine_type")
    .agg(
        distilbert_mean=(score_col, "mean"),
        distilbert_std=(score_col, "std"),
        distilbert_count=(score_col, "count"),
        stars_mean=("stars", "mean"),
    )
    .reset_index()
)
overall.to_csv("distilbert_results_overall.csv", index=False)

# tier-stratified results
by_tier = (
    export_df.groupby(["cuisine_type", "tier"])
    .agg(
        distilbert_mean=(score_col, "mean"),
        distilbert_std=(score_col, "std"),
        distilbert_count=(score_col, "count"),
        stars_mean=("stars", "mean"),
    )
    .reset_index()
)
by_tier.to_csv("distilbert_results_by_tier.csv", index=False)

# correlation + ANOVA + pairwise-count stats
r, _ = stats.pearsonr(export_df[score_col], export_df["stars"])

cuisine_groups = [
    g[score_col].values
    for _, g in export_df.groupby("cuisine_type")
    if len(g) >= MIN_REVIEWS
]
f_stat, p_val = stats.f_oneway(*cuisine_groups)

american = export_df[export_df["cuisine_type"] == "American"][score_col].values
sig_count = 0
for cuisine, group in export_df.groupby("cuisine_type"):
    if cuisine == "American" or len(group) < MIN_REVIEWS:
        continue
    other = group[score_col].values
    _, p_pair = stats.ttest_ind(american, other)
    if p_pair < 0.05:
        sig_count += 1

with open("distilbert_stats.txt", "w") as f:
    f.write(f"Pearson r: {r}\n")
    f.write(f"ANOVA F: {f_stat}\n")
    f.write(f"ANOVA p: {p_val}\n")
    f.write(f"Significant pairwise differences: {sig_count}/9\n")